# GP3 Experiment Runner
Configure the first cell, run the setup cell, then run the experiment. Default mode is **offline**: it fits saved real observations and recommends the next arm without making requests.

Only **Run the experiment** can make online pulls. The results and log cells always read local files, even when ONLINE is True. Each successful rerun of the online execution cell can spend another budget. Keep one driver for the whole group.

## 1. Settings
Choose uniform balanced exploration or UCB. Budget is a maximum number of completed online pulls in this invocation, not a training-data limit. Leave the group username blank to be prompted during an online run. The password is entered privately at runtime.

BUDGET=None uses the remaining group allowance, up to 1,000,000 total pulls. AUTO_TUNE enables a bounded local worker every TUNE_EVERY saved pulls, including an initial evaluation when all arms have data. Restart restores its saved schedule and coefficient. CVaR is a weighted average of the worst loss tail; see `../docs/tuning.md` for the rollout approximation and validation limits.


In [ ]:
# Main experiment settings
MODEL_NAME = "ucb"          # "uniform" or "ucb"
UCB_EXPLORATION = 0.005           # Positive; used only by UCB
BUDGET = None                   # None = all remaining; integer = additional pull cap
ONLINE = False                 # True enables real, budget-consuming requests
GROUP_USERNAME = ""            # Assigned group account; blank means prompt

# Files: relative paths are anchored to project3/program, not the kernel's cwd.
DB_PATH = "data/history.sqlite3"
LOG_PATH = None                 # None -> database name with .online.jsonl suffix

# Request settings (applied by the existing runner)
REQUEST_INTERVAL = 0.25         # Seconds between HTTP calls; minimum 0.25
REQUEST_TIMEOUT = 10.0          # Seconds, greater than 0 and at most 60
MAX_RETRIES = None              # Retry uncertain requests until stopped (or set a finite count)

# Local display settings
DISPLAY_ROWS = 20               # Most recent saved responses
LOG_ROWS = 20                   # Last log lines to inspect
# Optional local tuning; no real pulls are made by simulations.
AUTO_TUNE = False
TUNE_EVERY = 200000
TUNE_MAX_SECONDS = 600.0
TUNE_CVAR_LEVEL = 0.95
TUNE_STAGE_LOSS_ALLOWANCE = 0.0
TUNE_TAIL_LOSS_ALLOWANCE = 0.0
TARGET_TOTAL = None             # Optional absolute group counter cap
PROGRESS_EVERY = 1000


## 2. Setup
Run once after opening the notebook. This loads the existing framework and defines display helpers. No requests are made. Configuration is re-read each time you run or refresh a view, so changes in the first cell take effect after you execute that cell.

In [ ]:
import json
import sqlite3
import sys
from collections import deque
from contextlib import closing
from getpass import getpass
from html import escape
from pathlib import Path

from IPython.display import HTML, display

# Support starting Jupyter in program/, project3/, or the parent 5101/ folder.
candidates = [Path.cwd(), Path.cwd() / "program", Path.cwd() / "project3" / "program"]
PROGRAM_DIR = next(
    (path.resolve() for path in candidates
     if (path / "bandit" / "runner.py").is_file() and (path / "server_pull.py").is_file()),
    None,
)
if PROGRAM_DIR is None:
    raise RuntimeError("Open this notebook from project3/program or its project folders.")
if str(PROGRAM_DIR) not in sys.path:
    sys.path.insert(0, str(PROGRAM_DIR))

from bandit import UniformModel, UCBModel, TuningConfig, run_experiment
from bandit.audit import tail_lines
from bandit.client import validate_settings
from bandit.common import SERVER_URL, integer


def selected_model():
    if MODEL_NAME == "uniform":
        return UniformModel()
    if MODEL_NAME == "ucb":
        return UCBModel(exploration=UCB_EXPLORATION)
    raise ValueError("MODEL_NAME must be 'uniform' or 'ucb'.")


def resolve_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else PROGRAM_DIR / path).resolve()


def data_path():
    return resolve_path(DB_PATH)


def event_log_path():
    return resolve_path(LOG_PATH) if LOG_PATH is not None else data_path().with_suffix(".online.jsonl")


def resolved_settings():
    # Execution validation is deliberately separate from read-only display settings.
    if BUDGET is not None:
        integer(BUDGET, "BUDGET")
    if MAX_RETRIES is not None:
        integer(MAX_RETRIES, "MAX_RETRIES")
    if TARGET_TOTAL is not None:
        integer(TARGET_TOTAL, "TARGET_TOTAL", maximum=1000000)
    integer(PROGRESS_EVERY, "PROGRESS_EVERY", minimum=1)
    if type(AUTO_TUNE) is not bool:
        raise ValueError("AUTO_TUNE must be True or False.")
    tuning = TuningConfig(interval=TUNE_EVERY, max_seconds=TUNE_MAX_SECONDS,
                          cvar_level=TUNE_CVAR_LEVEL,
                          stage_loss_allowance=TUNE_STAGE_LOSS_ALLOWANCE,
                          tail_loss_allowance=TUNE_TAIL_LOSS_ALLOWANCE) if AUTO_TUNE else None
    if tuning and MODEL_NAME != "ucb":
        raise ValueError("Automatic tuning requires UCB.")
    if type(ONLINE) is not bool:
        raise ValueError("ONLINE must be True or False.")
    if not isinstance(GROUP_USERNAME, str):
        raise ValueError("GROUP_USERNAME must be a string.")
    validate_settings(REQUEST_INTERVAL, REQUEST_TIMEOUT)
    return dict(model=selected_model(), budget=BUDGET, online=ONLINE, db_path=data_path(),
                log_path=event_log_path(), interval=REQUEST_INTERVAL,
                timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES, server_url=SERVER_URL,
                target_total=TARGET_TOTAL, tuning=tuning, strict_history=True,
                progress_every=PROGRESS_EVERY,
                progress=lambda row: print("Saved", row["history_pulls"], "pulls; this run",
                                            row["completed_pulls"], flush=True))


def show_table(title, columns, rows):
    # Escape every data value so saved data cannot inject HTML.
    header = "".join("<th>" + escape(label) + "</th>" for key, label in columns)
    body = "".join(
        "<tr>" + "".join("<td>" + escape(str(row.get(key, ""))) + "</td>"
                        for key, label in columns) + "</tr>"
        for row in rows
    )
    if not rows:
        body = '<tr><td colspan="' + str(len(columns)) + '">No saved rows.</td></tr>'
    display(HTML(
        "<h3>" + escape(title) + "</h3>"
        '<div style="overflow-x:auto"><table style="border-collapse:collapse" border="1" cellpadding="7">'
        "<thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table></div>"
    ))


def percent(value):
    return "—" if value is None else "{:.2%}".format(value)


def load_snapshot(settings):
    model = settings["model"]
    identity, pending, recent = None, None, []
    last_saved_total = None
    db_path = settings["db_path"]
    if db_path.is_file():
        # mode=ro and one read transaction: views cannot modify history.
        with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as connection:
            connection.row_factory = sqlite3.Row
            connection.execute("BEGIN")
            row = connection.execute("SELECT value FROM metadata WHERE key='identity'").fetchone()
            identity = json.loads(row[0]) if row else None
            row = connection.execute("SELECT payload FROM pending WHERE singleton=1").fetchone()
            pending = json.loads(row[0]) if row else None
            records = connection.execute(
                "SELECT arm, COUNT(*) AS pulls, SUM(reward) AS signups FROM results GROUP BY arm"
            )
            model.fit_aggregates(dict(row) for row in records)
            if AUTO_TUNE and isinstance(model, UCBModel):
                saved = connection.execute("SELECT value FROM metadata WHERE key='tuning_state'").fetchone()
                if saved:
                    decision = json.loads(saved[0])
                    model.exploration = UCBModel(round(2*decision["selected_multiplier"]**2,12)).exploration
            recent = [dict(row) for row in connection.execute(
                "SELECT pull AS Pull, arm AS Arm, reward AS Reward, net_reward AS NetReward, "
                "saved_at, model FROM results ORDER BY pull DESC LIMIT ?", (DISPLAY_ROWS,)
            )]
            if recent:
                last_saved_total = recent[0]["NetReward"]
            recent.reverse()
    else:
        model.fit([])

    arms = model.summary()
    for row in arms:
        row["visitors"] = row["pulls"] * 100
        row["allocation_share"] = row["pulls"] / model.total_pulls if model.total_pulls else 0
    return dict(identity=identity, pending=pending, recent=recent, arms=arms,
                history_pulls=model.total_pulls, history_signups=sum(model.signups),
                last_saved_group_total=last_saved_total,
                next_arm=pending["arm"] if pending else model.select_arm(),
                model=model.name, parameters=model.parameters)


print("Ready. Settings will be read from the first cell when you run or refresh a view.")

## 3. Run the experiment
This is the only cell that can send online requests. With ONLINE=True and BUDGET=None (all remaining) or a positive budget, it asks for the group account and a hidden password, then calls the existing paced runner. With ONLINE=False or BUDGET=0, no credentials are requested.

The first interrupt requests a graceful stop after saving the in-flight response; a second can force interruption. If a request remains uncertain, use the viewing cells below to inspect committed results and pending recovery. Rerun with the same database/account to recover; do not switch databases to bypass a pending request.

In [ ]:
# Clear the previous summary so a failed rerun cannot display stale success.
last_run_summary = None
run_error = None


def execute_configured_run():
    settings = resolved_settings()  # Validate before prompting for credentials.
    username, password = None, None
    try:
        if settings["online"] and settings["budget"] != 0:
            username = GROUP_USERNAME.strip() or input("Group username: ").strip()
            if not username:
                raise ValueError("A group username is required.")
            password = getpass("Group password: ")
        return run_experiment(**settings, username=username, password=password)
    finally:
        password = None  # The secret never becomes a notebook global or output.


try:
    last_run_summary = execute_configured_run()
except KeyboardInterrupt:
    run_error = "KeyboardInterrupt"
    print("Interrupted. Saved results and pending request state are retained. Refresh the local views below.")
except Exception as error:
    run_error = type(error).__name__
    # Avoid persisting arbitrary server messages or tracebacks in notebook outputs.
    print("Run stopped (" + run_error + "). Check settings and refresh the saved results/log below.")

if last_run_summary is not None:
    items = [
        ("Mode", "ONLINE" if last_run_summary["online"] else "OFFLINE"),
        ("Model", last_run_summary["model"]),
        ("Parameters", json.dumps(last_run_summary["parameters"])),
        ("Requested online budget", last_run_summary["requested_budget"]),
        ("Completed pulls this run", last_run_summary["completed_pulls"]),
        ("Locally recorded pulls", last_run_summary["history_pulls"]),
        ("Locally recorded sign-ups", last_run_summary["history_signups"]),
        ("Next arm / pending recovery arm", last_run_summary["next_arm"]),
        ("Stop reason", last_run_summary["stop_reason"]),
        ("Database", last_run_summary["db_path"]),
    ]
    show_table("Run summary", [("label", "Item"), ("value", "Value")],
               [dict(label=label, value=value) for label, value in items])

## 4. Refresh saved results and arm statistics
Re-execute this cell whenever you want an updated local view. It opens SQLite **read-only** and rebuilds the selected model from saved observations. It never calls the server, including when ONLINE=True.

The local sign-up total includes only saved observations; it may differ from the last saved group cumulative total if historical responses are missing. Conversion rate uses sign-ups divided by visitors. Allocation share is the arm's share of locally recorded pulls. The next policy choice is not a claim that the arm is statistically best. UCB indices can exceed 1; unseen arms show a dash.

In [ ]:
integer(DISPLAY_ROWS, "DISPLAY_ROWS", minimum=1, maximum=1000)
view_settings = dict(db_path=data_path(), model=selected_model())
snapshot = load_snapshot(view_settings)
identity = snapshot["identity"] or {}
overview = [
    ("Selected model", snapshot["model"]),
    ("Parameters", json.dumps(snapshot["parameters"])),
    ("Saved group", identity.get("username", "No bound group yet")),
    ("Saved server", identity.get("server_url", "—")),
    ("Locally recorded pulls", snapshot["history_pulls"]),
    ("Locally recorded visitors", snapshot["history_pulls"] * 100),
    ("Locally recorded sign-ups", snapshot["history_signups"]),
    ("Last saved group cumulative sign-ups", snapshot["last_saved_group_total"]),
    ("Next policy choice / recovery arm", snapshot["next_arm"]),
    ("Database", str(view_settings["db_path"])),
]
show_table("Saved-history overview", [("label", "Item"), ("value", "Value")],
           [dict(label=label, value=value) for label, value in overview])

if snapshot["pending"]:
    show_table("Pending request — recover this before choosing a new arm",
               [("arm", "Arm"), ("expected_pull", "Expected pull"),
                ("request_id", "Request ID"), ("model", "Original model")],
               [snapshot["pending"]])

arm_rows = []
for row in snapshot["arms"]:
    arm_rows.append({
        **row,
        "conversion_rate_display": percent(row["conversion_rate"]),
        "allocation_display": percent(row["allocation_share"]),
        "score_display": "—" if row.get("score") is None else "{:.4f}".format(row["score"]),
        "next": "← next" if row["arm"] == snapshot["next_arm"] else "",
    })
columns = [("arm", "Arm"), ("pulls", "Pulls"), ("visitors", "Visitors"),
           ("signups", "Sign-ups"), ("conversion_rate_display", "Conversion rate"),
           ("allocation_display", "Allocation share")]
if snapshot["model"] == "ucb":
    columns.append(("score_display", "UCB index"))
columns.append(("next", "Policy choice"))
show_table("All 24 arms", columns, arm_rows)

show_table("Most recent saved pulls (up to " + str(DISPLAY_ROWS) + ")",
           [("Pull", "Pull index"), ("Arm", "Arm"), ("Reward", "Batch sign-ups"),
            ("NetReward", "Group cumulative sign-ups"), ("model", "Model"),
            ("saved_at", "Saved at (UTC)")], snapshot["recent"])

## 5. Refresh the online log
This cell reads only the local JSONL log. It shows the last configured number of lines, including request IDs, retries and returned rewards. A partial final line after interruption is reported and skipped. No log exists until an online run has created one.

In [ ]:
integer(LOG_ROWS, "LOG_ROWS", minimum=1, maximum=1000)
log_file = event_log_path()
log_rows = []
skipped = 0

if log_file.is_file():
    # Bounded display memory even for long experiments.
    tail = tail_lines(log_file, LOG_ROWS)
    for line in tail:
        try:
            event = json.loads(line)
            if not isinstance(event, dict):
                raise ValueError("Invalid event")
        except (ValueError, json.JSONDecodeError):
            skipped += 1
            continue
        request = event.get("request") or event.get("pending_request") or {}
        result = event.get("result") or {}
        log_rows.append({
            "timestamp": event.get("timestamp", ""),
            "event": event.get("event", ""),
            "model": request.get("model", event.get("model", "")),
            "arm": request.get("arm", result.get("Arm", "")),
            "pull": request.get("expected_pull", result.get("Pull", "")),
            "attempt": event.get("attempt", ""),
            "reward": result.get("Reward", ""),
            "total": result.get("NetReward", ""),
            "request_id": request.get("request_id", ""),
            "detail": event.get("stop_reason", event.get("reason", event.get("error_type", ""))),
        })
else:
    print("No online log yet:", log_file)
if skipped:
    print("Skipped", skipped, "incomplete or invalid log line(s).")

show_table("Recent online events", [("timestamp", "Time (UTC)"), ("event", "Event"),
           ("model", "Model"), ("arm", "Arm"), ("pull", "Pull"), ("attempt", "Attempt"),
           ("reward", "Reward"), ("total", "Group total"), ("request_id", "Request ID"),
           ("detail", "Detail")], log_rows)

## Exporting for review or submission
Save the notebook to retain its displayed tables. Export to HTML using saved outputs only; do **not** enable execution during export. For example, from the program directory run: jupyter nbconvert --to html "GP3 Run.ipynb".

Never put the password in a cell. Keep the database and log together, use one driver for the group, and preserve the pending state if recovery is needed. The supplied GP3 Starter notebook remains unchanged.